In [ ]:
"""
Chapter 6 — Figures Generator
═══════════════════════════════════════════════════════════════════════════════
Project  : Automatic Road Network Extraction with Connectivity and
           Topology Refinement from High-Resolution Satellite Imagery
Dataset  : SpaceNet 5 (SN5) — AOI 8, Mumbai, India
Institute: VR Siddhartha Engineering College, Vijayawada
Team     : Murala Leela Kartheek (228W1A0529)
           Mallam Manoj         (228W1A0525)
           Ch Devarshisai       (228W1A0510)
Guide    : Dr. G. Kranthi Kumar

Generates all figures required for Chapter 6:
  fig6_1_input_groundtruth.png   — Input + GeoJSON GT overlay
  fig6_2_dataset_sample.png      — Satellite tile + binary mask
  fig6_3_segmentation_output.png — Input + GT + Predicted mask
  fig6_5_connectivity.png        — Before / after connectivity fix
  fig6_6_centerlines.png         — Mask + skeletonized centerlines
  fig6_7_topology.png            — Vector network before / after refinement
  fig6_8_confusion_matrix.png    — 2×2 confusion matrix heatmap
  fig6_9_pipeline_output.png     — Full 5-panel end-to-end output
═══════════════════════════════════════════════════════════════════════════════
"""

import math, json, random
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
from pathlib import Path
from skimage.morphology import skeletonize
from skimage.measure    import label as sklabel
from scipy.spatial      import cKDTree
import warnings
warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    "val_npz":         "/home/jupyter-228w1a0529/Major/Dataset-1/processed/val.npz",
    "centerlines_npz": "/home/jupyter-228w1a0529/Major/Dataset-1/processed/centerlines/centerlines.npz",
    "vectors_dir":     "/home/jupyter-228w1a0529/Major/Dataset-1/processed/vectors",
    "output_dir":      "/home/jupyter-228w1a0529/Major/chapter6_figures",

    # ImageNet denorm
    "mean": np.array([0.485, 0.456, 0.406]),
    "std":  np.array([0.229, 0.224, 0.225]),

    # Figure DPI — 200 for publication quality
    "dpi": 200,

    # Random seed for reproducible patch selection
    "seed": 42,

    # Morphological params (must match Module 3)
    "morph_kernel_small": 5,
    "morph_kernel_large": 11,
    "reconnect_delta":    25,
}

# Module 3 accuracy values (from your actual run)
MODULE3_METRICS = {
    "raw_iou":   0.6818,
    "final_iou": 0.6815,
    "raw_f1":    0.7917,
    "final_f1":  0.7916,
    "raw_prec":  0.7885,
    "final_prec":0.7872,
    "raw_rec":   0.8079,
    "final_rec": 0.8088,
}


# ═══════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def denorm(img_chw, config=CONFIG):
    """(3,H,W) normalised → (H,W,3) float [0,1]."""
    return (np.transpose(img_chw, (1,2,0)) * config["std"]
            + config["mean"]).clip(0, 1)


def load_data(config=CONFIG):
    """Load val.npz and centerlines.npz."""
    print("[INFO] Loading data...")
    v  = np.load(config["val_npz"])
    cl = np.load(config["centerlines_npz"])
    return (v["images"], v["masks"].astype(np.uint8),
            cl["raw_preds"], cl["eval_masks"], cl["centerlines"])


def geojson_to_graph(gj):
    G = nx.Graph()
    for feat in gj.get("features", []):
        coords = feat["geometry"]["coordinates"]
        if len(coords) < 2: continue
        pts = [(int(c[1]), int(c[0])) for c in coords]
        for i in range(len(pts)-1):
            u, v = pts[i], pts[i+1]
            G.add_node(u, y=u[0], x=u[1])
            G.add_node(v, y=v[0], x=v[1])
            l = math.sqrt((u[0]-v[0])**2+(u[1]-v[1])**2)
            G.add_edge(u, v, length=l, geometry=[u,v])
    return G


def load_graph(idx, config=CONFIG):
    p = Path(config["vectors_dir"]) / f"patch_{idx:04d}.geojson"
    if p.exists():
        with open(p) as f: return geojson_to_graph(json.load(f))
    return nx.Graph()


def draw_graph_on_img(img_hwc, G, edge_color=(0.0,0.5,1.0),
                       node_size=2, edge_thickness=2):
    """Draw NetworkX graph on image copy."""
    ov   = img_hwc.copy()
    H, W = ov.shape[:2]
    for u, v, data in G.edges(data=True):
        geom = data.get("geometry", [u, v])
        for i in range(len(geom)-1):
            r1,c1 = int(geom[i][0]),   int(geom[i][1])
            r2,c2 = int(geom[i+1][0]), int(geom[i+1][1])
            if 0<=r1<H and 0<=c1<W and 0<=r2<H and 0<=c2<W:
                cv2.line(ov,(c1,r1),(c2,r2),edge_color,edge_thickness)
    for node in G.nodes():
        r, c = int(node[0]), int(node[1])
        if 0<=r<H and 0<=c<W:
            col = (1.0,0.1,0.1) if G.degree(node)>=3 else (0.1,1.0,0.1)
            cv2.circle(ov,(c,r),node_size,col,-1)
    return ov


def pick_good_patches(images, gt_masks, raw_preds,
                       n=8, seed=42, config=CONFIG):
    """
    Select patches that:
      1. Have a reasonable road content (gt road pixels > 1%)
      2. Have non-empty predictions
    Returns list of indices.
    """
    rng = np.random.default_rng(seed)
    road_frac = gt_masks.mean(axis=(1,2))
    pred_frac = raw_preds.mean(axis=(1,2))
    # patches where GT roads > 2% and pred is non-empty
    candidates = np.where((road_frac > 0.02) & (pred_frac > 0.005))[0]
    if len(candidates) < n:
        candidates = np.where(road_frac > 0.01)[0]
    chosen = rng.choice(candidates, min(n, len(candidates)), replace=False)
    return sorted(chosen.tolist())


def apply_closing(mask, ks, it=1):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(ks,ks))
    return cv2.morphologyEx(mask.astype(np.uint8),cv2.MORPH_CLOSE,k,
                             iterations=it)


def reconnect(mask, delta=25):
    from skimage.morphology import skeletonize as skel_fn
    skel = skel_fn(mask.astype(bool)).astype(np.uint8)
    k    = np.ones((3,3),dtype=np.float32); k[1,1]=0
    nc   = cv2.filter2D(skel.astype(np.float32),-1,k)
    ep   = np.column_stack(np.where((skel==1)&(nc==1)))
    res  = mask.copy()
    if len(ep) < 2: return res
    used = set()
    for i in range(len(ep)):
        if i in used: continue
        r1,c1 = ep[i]
        bd,bj = delta+1,-1
        for j in range(i+1,len(ep)):
            if j in used: continue
            r2,c2 = ep[j]
            d = math.sqrt((r1-r2)**2+(c1-c2)**2)
            if d<bd: bd,bj=d,j
        if bj>=0:
            r2,c2=ep[bj]
            cv2.line(res,(c1,r1),(c2,r2),1,3)
            used.add(i); used.add(bj)
    return res


def build_raw_graph(skeleton):
    """Minimal raster→graph for topology before/after comparison."""
    G = nx.Graph()
    if skeleton.sum()==0: return G
    pts = list(map(tuple,np.argwhere(skeleton>0)))
    G.add_nodes_from(pts)
    H,W = skeleton.shape
    for r,c in pts:
        for dr in [-1,0,1]:
            for dc in [-1,0,1]:
                if dr==0 and dc==0: continue
                nr,nc=r+dr,c+dc
                if 0<=nr<H and 0<=nc<W and skeleton[nr,nc]>0:
                    G.add_edge((r,c),(nr,nc),
                               weight=float(math.sqrt(dr**2+dc**2)),
                               geometry=[(r,c),(nr,nc)])
    return G


def save(fig, path, dpi):
    fig.savefig(path, dpi=dpi, bbox_inches="tight",
                facecolor="white", edgecolor="none")
    plt.close(fig)
    print(f"[INFO] ✅ Saved → {path}")


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.1 — Input + GeoJSON GT Overlay
# ═══════════════════════════════════════════════════════════════════════════

def fig6_1(images, gt_masks, idx, out_dir, config=CONFIG):
    """
    Left:  raw PS-RGB satellite image
    Right: same image with GT road mask overlaid as colored lines
    """
    img = denorm(images[idx], config)
    gt  = gt_masks[idx].astype(np.uint8)

    # Overlay GT as colored roads on image
    overlay = img.copy()
    # Dilate slightly to make roads visible
    k   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    gt_d= cv2.dilate(gt, k)
    overlay[gt_d.astype(bool)] = [1.0, 0.85, 0.0]  # yellow

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    fig.patch.set_facecolor("white")

    axes[0].imshow(img)
    axes[0].set_title("(a) PS-RGB Satellite Image", fontsize=13,
                       fontweight="bold", pad=8)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title("(b) Ground Truth Road Annotation",
                       fontsize=13, fontweight="bold", pad=8)
    yp = mpatches.Patch(color=(1.0,0.85,0.0), label="Annotated road")
    axes[1].legend(handles=[yp], loc="lower right", fontsize=10,
                    framealpha=0.85)
    axes[1].axis("off")

    fig.suptitle(
        "Figure 6.1: Sample input PS-RGB satellite image and corresponding\n"
        "ground truth road annotation from SpaceNet AOI 8 – Mumbai dataset.",
        fontsize=10, style="italic", y=0.02
    )
    plt.tight_layout(rect=[0,0.07,1,1])
    save(fig, out_dir/"fig6_1_input_groundtruth.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.2 — Dataset Sample
# ═══════════════════════════════════════════════════════════════════════════

def fig6_2(images, gt_masks, idx, out_dir, config=CONFIG):
    """
    Left:  512×512 satellite tile
    Right: binary road mask (white roads on black)
    """
    img = denorm(images[idx], config)
    gt  = gt_masks[idx].astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    fig.patch.set_facecolor("white")

    axes[0].imshow(img)
    axes[0].set_title("(a) Input PS-RGB Satellite Tile (512×512 px)",
                       fontsize=12, fontweight="bold", pad=8)
    axes[0].axis("off")

    axes[1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("(b) Binary Road Mask\n(white=road, black=background)",
                       fontsize=12, fontweight="bold", pad=8)
    axes[1].axis("off")

    fig.suptitle(
        "Figure 6.2: Representative sample from the prepared dataset showing\n"
        "(a) input PS-RGB satellite image tile and (b) corresponding binary\n"
        "road mask generated from GeoJSON road annotations.",
        fontsize=10, style="italic", y=0.02
    )
    plt.tight_layout(rect=[0,0.1,1,1])
    save(fig, out_dir/"fig6_2_dataset_sample.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.3 — Segmentation Output
# ═══════════════════════════════════════════════════════════════════════════

def fig6_3(images, gt_masks, raw_preds, idx, out_dir, config=CONFIG):
    """
    Three panels: Input | GT | Predicted mask
    """
    img  = denorm(images[idx], config)
    gt   = gt_masks[idx].astype(np.uint8)
    pred = raw_preds[idx].astype(np.uint8)

    inter = float((pred.astype(bool) & gt.astype(bool)).sum())
    union = float((pred.astype(bool) | gt.astype(bool)).sum())
    iou   = inter / (union + 1e-6)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
    fig.patch.set_facecolor("white")

    axes[0].imshow(img)
    axes[0].set_title("(a) Input Satellite Image", fontsize=12,
                       fontweight="bold", pad=8)
    axes[0].axis("off")

    axes[1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("(b) Ground Truth Road Mask", fontsize=12,
                       fontweight="bold", pad=8)
    axes[1].axis("off")

    axes[2].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"(c) Predicted Road Mask\n(IoU = {iou:.3f})",
                       fontsize=12, fontweight="bold", pad=8)
    axes[2].axis("off")

    fig.suptitle(
        "Figure 6.3: Sample road segmentation output showing (a) input satellite image tile,\n"
        "(b) ground truth binary road mask, and (c) predicted binary road mask from the\n"
        "proposed DeepLabV3+ model with ECA-Net channel attention.",
        fontsize=10, style="italic", y=0.02
    )
    plt.tight_layout(rect=[0,0.1,1,1])
    save(fig, out_dir/"fig6_3_segmentation_output.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.5 — Connectivity Enhancement (before / after)
# ═══════════════════════════════════════════════════════════════════════════

def fig6_5(raw_preds, idx, out_dir, config=CONFIG):
    """
    Left:  raw prediction (fragmented)
    Right: after morphological closing + reconnection (continuous)
    """
    raw = raw_preds[idx].astype(np.uint8)

    # Apply Module 3 closing pipeline
    closed = apply_closing(raw, config["morph_kernel_small"])
    closed = apply_closing(closed, config["morph_kernel_large"])
    fixed  = reconnect(closed, config["reconnect_delta"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    fig.patch.set_facecolor("white")

    axes[0].imshow(raw, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("(a) Before Post-processing\n"
                       "(fragmented road segments)",
                       fontsize=12, fontweight="bold", pad=8)
    axes[0].axis("off")

    axes[1].imshow(fixed, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("(b) After Connectivity Enhancement\n"
                       "(morphological closing + gap bridging)",
                       fontsize=12, fontweight="bold", pad=8)
    axes[1].axis("off")

    fig.suptitle(
        "Figure 6.5: Binary road mask (a) before and (b) after connectivity enhancement\n"
        "through morphological closing and Euclidean distance-based gap bridging.",
        fontsize=10, style="italic", y=0.02
    )
    plt.tight_layout(rect=[0,0.08,1,1])
    save(fig, out_dir/"fig6_5_connectivity.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.6 — Centerline Extraction
# ═══════════════════════════════════════════════════════════════════════════

def fig6_6(images, eval_masks, centerlines, idx, out_dir, config=CONFIG):
    """
    Left:  connectivity-enhanced mask
    Right: 1-pixel skeleton overlaid in red on the mask
    """
    img = denorm(images[idx], config)
    em  = eval_masks[idx].astype(np.uint8)
    cl  = centerlines[idx].astype(np.uint8)

    # Red centerline on mask as RGB
    mask_rgb = np.stack([em, em, em], axis=-1).astype(float)
    k  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    cl_d = cv2.dilate(cl, k).astype(bool)
    overlay = mask_rgb.copy()
    overlay[cl_d] = [1.0, 0.0, 0.0]   # red

    # Also show on satellite image
    sat_ov = img.copy()
    sat_ov[cl_d] = [1.0, 0.15, 0.15]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
    fig.patch.set_facecolor("white")

    axes[0].imshow(em, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("(a) Connectivity-Enhanced Mask",
                       fontsize=12, fontweight="bold", pad=8)
    axes[0].axis("off")

    axes[1].imshow(overlay, vmin=0, vmax=1)
    axes[1].set_title("(b) Skeletonized Centerlines\n(red) on mask",
                       fontsize=12, fontweight="bold", pad=8)
    rp = mpatches.Patch(color=(1,0,0), label="1-px centerline")
    axes[1].legend(handles=[rp], loc="lower right", fontsize=9,
                    framealpha=0.85)
    axes[1].axis("off")

    axes[2].imshow(sat_ov)
    axes[2].set_title("(c) Centerlines on Satellite Image",
                       fontsize=12, fontweight="bold", pad=8)
    axes[2].legend(handles=[rp], loc="lower right", fontsize=9,
                    framealpha=0.85)
    axes[2].axis("off")

    fig.suptitle(
        "Figure 6.6: Extracted one-pixel-wide road centerlines obtained through skeletonization\n"
        "of the connectivity-enhanced binary road mask for a representative test tile.",
        fontsize=10, style="italic", y=0.02
    )
    plt.tight_layout(rect=[0,0.08,1,1])
    save(fig, out_dir/"fig6_6_centerlines.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.7 — Vectorization and Topology Refinement
# ═══════════════════════════════════════════════════════════════════════════

def fig6_7(images, centerlines, idx, out_dir, config=CONFIG):
    """
    Left:  raw dense graph (before refinement) — spurious short edges visible
    Right: topology-refined clean graph
    """
    img = denorm(images[idx], config)
    cl  = centerlines[idx].astype(np.uint8)

    # Raw graph — all skeleton pixels connected
    G_raw = build_raw_graph(cl)

    # Refined graph — load from saved GeoJSON
    G_ref = load_graph(idx, config)

    raw_ov = draw_graph_on_img(img, G_raw,
                                edge_color=(1.0,0.4,0.0),
                                node_size=1, edge_thickness=1)
    ref_ov = draw_graph_on_img(img, G_ref,
                                edge_color=(0.0,0.5,1.0),
                                node_size=2, edge_thickness=2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor("white")

    axes[0].imshow(raw_ov)
    axes[0].set_title(
        f"(a) Before Topology Refinement\n"
        f"V={G_raw.number_of_nodes():,}  E={G_raw.number_of_edges():,}",
        fontsize=12, fontweight="bold", pad=8
    )
    orp = mpatches.Patch(color=(1.0,0.4,0.0), label="Raw edge")
    axes[0].legend(handles=[orp], loc="lower right", fontsize=9,
                    framealpha=0.85)
    axes[0].axis("off")

    axes[1].imshow(ref_ov)
    axes[1].set_title(
        f"(b) After Topology Refinement\n"
        f"V={G_ref.number_of_nodes()}  E={G_ref.number_of_edges()}",
        fontsize=12, fontweight="bold", pad=8
    )
    bp  = mpatches.Patch(color=(0,0.5,1),   label="Road edge")
    gp  = mpatches.Patch(color=(0.1,1,0.1), label="Node")
    rp  = mpatches.Patch(color=(1,0.1,0.1), label="Junction")
    axes[1].legend(handles=[bp,gp,rp], loc="lower right", fontsize=9,
                    framealpha=0.85)
    axes[1].axis("off")

    fig.suptitle(
        "Figure 6.7: Vectorized road network (a) before and (b) after topology refinement,\n"
        "illustrating the removal of spurious edges, merging of nearby nodes, and\n"
        "elimination of isolated graph components.",
        fontsize=10, style="italic", y=0.02
    )
    plt.tight_layout(rect=[0,0.1,1,1])
    save(fig, out_dir/"fig6_7_topology.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.8 — Confusion Matrix
# ═══════════════════════════════════════════════════════════════════════════

def fig6_8(gt_masks, raw_preds, out_dir, config=CONFIG):
    """
    Compute aggregate pixel-level confusion matrix across all val patches.
    Display as seaborn heatmap with counts and percentages.
    """
    print("[INFO] Computing confusion matrix (all val patches)...")
    TP = FP = FN = TN = 0
    for gt, pred in zip(gt_masks, raw_preds):
        g = gt.astype(bool).ravel()
        p = pred.astype(bool).ravel()
        TP += int(( p &  g).sum())
        FP += int(( p & ~g).sum())
        FN += int((~p &  g).sum())
        TN += int((~p & ~g).sum())

    total  = TP + FP + FN + TN
    cm     = np.array([[TP, FP],
                        [FN, TN]])
    cm_pct = cm / total * 100

    # Labels with counts and percentages
    labels = np.array([
        [f"TP\n{TP:,}\n({cm_pct[0,0]:.1f}%)",
         f"FP\n{FP:,}\n({cm_pct[0,1]:.1f}%)"],
        [f"FN\n{FN:,}\n({cm_pct[1,0]:.1f}%)",
         f"TN\n{TN:,}\n({cm_pct[1,1]:.1f}%)"]
    ])

    precision = TP / (TP+FP+1e-6)
    recall    = TP / (TP+FN+1e-6)
    f1        = 2*precision*recall/(precision+recall+1e-6)
    iou_val   = TP / (TP+FP+FN+1e-6)

    fig, ax = plt.subplots(figsize=(8, 6.5))
    fig.patch.set_facecolor("white")

    sns.heatmap(cm, annot=labels, fmt="", cmap="Blues",
                xticklabels=["Predicted\nPositive (Road)",
                              "Predicted\nNegative (Background)"],
                yticklabels=["Actual\nPositive (Road)",
                              "Actual\nNegative (Background)"],
                linewidths=1.5, linecolor="white",
                annot_kws={"size":13,"weight":"bold"},
                ax=ax, cbar_kws={"label":"Pixel count"})

    ax.set_title("Pixel-Level Confusion Matrix\n"
                  f"Precision={precision:.3f}  Recall={recall:.3f}  "
                  f"F1={f1:.3f}  IoU={iou_val:.3f}",
                  fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
    ax.set_ylabel("Actual Label",    fontsize=12, fontweight="bold")
    ax.tick_params(labelsize=11)

    fig.suptitle(
        "Figure 6.8: Confusion matrix of the road segmentation model\n"
        "evaluated on the SpaceNet AOI 8 – Mumbai validation set.",
        fontsize=10, style="italic", y=0.01
    )
    plt.tight_layout(rect=[0,0.06,1,1])
    save(fig, out_dir/"fig6_8_confusion_matrix.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# FIG 6.9 — End-to-End Pipeline Output (5-panel)
# ═══════════════════════════════════════════════════════════════════════════

def fig6_9(images, gt_masks, raw_preds, eval_masks,
            centerlines, idx, out_dir, config=CONFIG):
    """
    5 panels in one row:
    (a) Input  (b) Predicted mask  (c) Connectivity mask
    (d) Centerlines  (e) Vector road network on satellite
    """
    img  = denorm(images[idx], config)
    gt   = gt_masks[idx].astype(np.uint8)
    pred = raw_preds[idx].astype(np.uint8)
    em   = eval_masks[idx].astype(np.uint8)
    cl   = centerlines[idx].astype(np.uint8)
    G    = load_graph(idx, config)

    # Centerline red overlay on satellite
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    cl_d = cv2.dilate(cl, k).astype(bool)
    cl_ov= img.copy(); cl_ov[cl_d] = [1.0,0.15,0.15]

    # Graph overlay on satellite
    gr_ov = draw_graph_on_img(img, G,
                               edge_color=(0.0,0.5,1.0),
                               node_size=2, edge_thickness=2)

    fig, axes = plt.subplots(1, 5, figsize=(25, 5.5))
    fig.patch.set_facecolor("white")

    panels = [
        (img,    None,   "(a) Input\nPS-RGB Image"),
        (pred,   "gray", "(b) Predicted\nRoad Mask"),
        (em,     "gray", "(c) Connectivity-Enhanced\nRoad Mask"),
        (cl_ov,  None,   "(d) Skeletonized\nCenterlines"),
        (gr_ov,  None,   "(e) GIS Vector\nRoad Network"),
    ]

    for ax, (data, cmap, title) in zip(axes, panels):
        ax.imshow(data, cmap=cmap, vmin=0 if cmap else None,
                   vmax=1 if cmap else None)
        ax.set_title(title, fontsize=10.5, fontweight="bold", pad=7)
        ax.axis("off")

    # Add legend to last panel
    bp = mpatches.Patch(color=(0,.5,1),   label="Road edge")
    gp = mpatches.Patch(color=(0.1,1,0.1),label="Node")
    rp = mpatches.Patch(color=(1,0.1,0.1),label="Junction")
    axes[4].legend(handles=[bp,gp,rp], loc="lower right",
                    fontsize=8, framealpha=0.85)

    fig.suptitle(
        "Figure 6.9: End-to-end pipeline output for a representative SpaceNet AOI 8 – Mumbai test tile showing\n"
        "(a) input PS-RGB satellite image, (b) predicted binary road mask, (c) connectivity-enhanced road mask,\n"
        "(d) skeletonized road centerlines, and (e) final GIS-ready vector road network overlaid on satellite image.",
        fontsize=10, style="italic", y=0.01
    )
    plt.tight_layout(rect=[0,0.1,1,1])
    save(fig, out_dir/"fig6_9_pipeline_output.png", config["dpi"])


# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

def generate_all_figures(config=CONFIG):
    print("="*62)
    print(" Chapter 6 — Figure Generator")
    print("="*62)

    out_dir = Path(config["output_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Output dir: {out_dir}/\n")

    # Load all data
    images, gt_masks, raw_preds, eval_masks, centerlines = \
        load_data(config)
    print(f"[INFO] Dataset: {len(images)} validation patches")

    # Pick good representative patches
    idxs = pick_good_patches(images, gt_masks, raw_preds,
                               n=20, seed=config["seed"], config=config)
    print(f"[INFO] Selected {len(idxs)} patches: {idxs[:8]}...")

    # One primary patch for single-patch figures
    # Pick the one with highest road content among selected
    road_fracs = [gt_masks[i].mean() for i in idxs]
    primary_idx = idxs[int(np.argmax(road_fracs))]

    # Secondary idx for variety
    secondary_idx = idxs[1] if len(idxs) > 1 else idxs[0]

    print(f"[INFO] Primary patch: {primary_idx} "
          f"(road={gt_masks[primary_idx].mean()*100:.1f}%)")
    print(f"[INFO] Secondary patch: {secondary_idx}")
    print()

    # Generate each figure
    print("[INFO] Generating Fig 6.1 — Input + GT overlay...")
    fig6_1(images, gt_masks, primary_idx, out_dir, config)

    print("[INFO] Generating Fig 6.2 — Dataset sample...")
    fig6_2(images, gt_masks, secondary_idx, out_dir, config)

    print("[INFO] Generating Fig 6.3 — Segmentation output...")
    fig6_3(images, gt_masks, raw_preds, primary_idx, out_dir, config)

    print("[INFO] Generating Fig 6.5 — Connectivity enhancement...")
    fig6_5(raw_preds, primary_idx, out_dir, config)

    print("[INFO] Generating Fig 6.6 — Centerline extraction...")
    fig6_6(images, eval_masks, centerlines, primary_idx, out_dir, config)

    print("[INFO] Generating Fig 6.7 — Topology refinement...")
    fig6_7(images, centerlines, primary_idx, out_dir, config)

    print("[INFO] Generating Fig 6.8 — Confusion matrix...")
    fig6_8(gt_masks, raw_preds, out_dir, config)

    print("[INFO] Generating Fig 6.9 — Full pipeline output...")
    fig6_9(images, gt_masks, raw_preds, eval_masks,
            centerlines, primary_idx, out_dir, config)

    print(f"\n{'='*62}")
    print(f" ✅  All 8 figures saved to: {out_dir}/")
    print(f"{'='*62}")
    print(f"  fig6_1_input_groundtruth.png")
    print(f"  fig6_2_dataset_sample.png")
    print(f"  fig6_3_segmentation_output.png")
    print(f"  fig6_5_connectivity.png")
    print(f"  fig6_6_centerlines.png")
    print(f"  fig6_7_topology.png")
    print(f"  fig6_8_confusion_matrix.png")
    print(f"  fig6_9_pipeline_output.png")
    print(f"{'='*62}")
    print(f"\n  📋 Use these directly in your Chapter 6 report!")


if __name__ == "__main__":
    # !pip install seaborn --break-system-packages
    generate_all_figures(CONFIG)

 Chapter 6 — Figure Generator
[INFO] Output dir: /home/jupyter-228w1a0529/Major/chapter6_figures/

[INFO] Loading data...
[INFO] Dataset: 3971 validation patches
[INFO] Selected 20 patches: [358, 369, 389, 530, 835, 1722, 1746, 1813]...
[INFO] Primary patch: 2765 (road=5.0%)
[INFO] Secondary patch: 369

[INFO] Generating Fig 6.1 — Input + GT overlay...
[INFO] ✅ Saved → /home/jupyter-228w1a0529/Major/chapter6_figures/fig6_1_input_groundtruth.png
[INFO] Generating Fig 6.2 — Dataset sample...
